# SSTW S1 finite propagation-sensitivity screen

> **选择 L4 GPU 后全部运行。** 该 Notebook 执行预冻结层级式 relation→block→velocity screen；不运行 VAE、MP4 或 S2。

METHOD_ONLY / DIAGNOSTIC_ONLY. Full non-degenerate path is 170 transformer calls; deterministic invalid-weight or no-usable-construction paths stop earlier without retry.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
REPOSITORY_URL = 'https://github.com/RICHAAARC/SC-SSTW-Feasibility.git'
AUTHORIZED_REF = 'f11e4fbfd084809d294b64c8ac425409367aedcc'
RUN_ID = 'ff43e8be9475cbdb'
DRIVE_OUTPUT_ROOT = '/content/drive/MyDrive/SC-SSTW-Feasibility/s1-propagation-multipair-unipc-fork-ff43e8be9475cbdb'
AUTHORIZE_EXECUTION = True
AUTHORIZE_DRIVE_IO = True
AUTHORITY_COMMIT = '0f85bd70de6f042c68560ed348722fc4fbd112e9'
AUTHORITY_TREE = 'd791cff7b5cbebb6ec690f16196bde9ee6d50bd1'
AUTHORITY_RAW_SHA256 = '85c32f49897a6b7c6248d5fd6fe4c7a307b4b841dc7388551d095854b6189292'
MODEL_ID = 'Wan-AI/Wan2.1-T2V-1.3B-Diffusers'
MODEL_REVISION = '0fad780a534b6463e45facd96134c9f345acfa5b'
EXPECTED_CONFIG_SHA256 = '7d01c7591753293518e821265dda6005070763373511e808cdd2dcad3d56c560'


In [ ]:
import subprocess
gpu_probe = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], check=False, capture_output=True, text=True)
print('GPU diagnostic:', gpu_probe.stdout.strip() or gpu_probe.stderr.strip() or 'unavailable')
print('L4 is supported; acceptance requires CUDA and BF16 capability only.')


In [ ]:
from pathlib import Path
import hashlib, json, re, shutil, subprocess, sys, zipfile

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def require_absent(*paths):
    for path in paths:
        if path.exists() or path.is_symlink(): raise RuntimeError(f'refusing to overwrite: {path}')

if not AUTHORIZE_EXECUTION or not AUTHORIZE_DRIVE_IO:
    raise RuntimeError('explicit execution and Drive acknowledgements are required')
if not re.fullmatch(r'[0-9a-f]{40}', AUTHORIZED_REF) or not re.fullmatch(r'[0-9a-f]{16}', RUN_ID):
    raise RuntimeError('exact ref and run id are required')
WORK = Path('/content') / f'sstw-s1-propagation-source-{RUN_ID}'
LOCAL_ROOT = Path('/content') / f'sstw-s1-propagation-run-{RUN_ID}'
OUTPUT = LOCAL_ROOT / 'output'
LOG = Path('/content') / f'sstw-s1-propagation-log-{RUN_ID}'
BUNDLE = Path('/content') / f'sstw-s1-propagation-bundle-{RUN_ID}'
ARCHIVE = Path('/content') / f'sstw-s1-propagation-{RUN_ID}.zip'
SIDECAR = Path('/content') / f'sstw-s1-propagation-{RUN_ID}.zip.sha256.json'
DRIVE_ROOT = Path(DRIVE_OUTPUT_ROOT)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
if DRIVE_ROOT.is_symlink() or not DRIVE_ROOT.is_dir(): raise RuntimeError('Drive root invalid')
DRIVE_ARCHIVE, DRIVE_SIDECAR = DRIVE_ROOT / ARCHIVE.name, DRIVE_ROOT / SIDECAR.name
require_absent(WORK, LOCAL_ROOT, LOG, BUNDLE, ARCHIVE, SIDECAR, DRIVE_ARCHIVE, DRIVE_SIDECAR)
LOG.mkdir()
runner_started = False
completed = audit = caught = None
try:
    with (LOG / 'clone.stdout').open('xb') as out, (LOG / 'clone.stderr').open('xb') as err:
        subprocess.run(['git','clone','--no-checkout',REPOSITORY_URL,str(WORK)], check=True, stdout=out, stderr=err)
    subprocess.run(['git','checkout','--detach',AUTHORIZED_REF], cwd=WORK, check=True, capture_output=True)
    head = subprocess.run(['git','rev-parse','HEAD'], cwd=WORK, check=True, capture_output=True, text=True).stdout.strip()
    dirty = subprocess.run(['git','status','--porcelain=v1','--untracked-files=all'], cwd=WORK, check=True, capture_output=True, text=True).stdout
    if head != AUTHORIZED_REF or dirty: raise RuntimeError('clean exact-ref checkout failed')
    tree = subprocess.run(['git','rev-parse',AUTHORITY_COMMIT+'^{tree}'], cwd=WORK, check=True, capture_output=True, text=True).stdout.strip()
    authority = subprocess.run(['git','show',AUTHORITY_COMMIT+':SSTW_METHOD_AUTHORITY.md'], cwd=WORK, check=True, capture_output=True).stdout
    ancestor = subprocess.run(['git','merge-base','--is-ancestor',AUTHORITY_COMMIT,AUTHORIZED_REF], cwd=WORK).returncode == 0
    if tree != AUTHORITY_TREE or hashlib.sha256(authority).hexdigest() != AUTHORITY_RAW_SHA256 or not ancestor: raise RuntimeError('authority identity mismatch')
    config_path = WORK / 'configs/s1_propagation_sensitivity_multipair.json'
    if sha256_file(config_path) != EXPECTED_CONFIG_SHA256: raise RuntimeError('config identity mismatch')
    expected_run = hashlib.sha256(('SSTW-S1-PROPAGATION-MULTIPAIR:' + AUTHORIZED_REF + ':' + EXPECTED_CONFIG_SHA256).encode()).hexdigest()[:16]
    if RUN_ID != expected_run: raise RuntimeError('run id binding mismatch')
    locked = ['accelerate==1.4.0','diffusers==0.35.2','ftfy==6.3.1','huggingface_hub==0.35.3','numpy==1.26.4','safetensors==0.5.3','transformers==4.49.0']
    subprocess.run([sys.executable,'-m','pip','install',*locked], check=True, stdout=(LOG/'pip.stdout').open('xb'), stderr=(LOG/'pip.stderr').open('xb'))
    import torch
    if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported(): raise RuntimeError('CUDA and BF16 capability are required')
    print('Runtime diagnostic:', {'torch':torch.__version__,'cuda':torch.version.cuda,'gpu':torch.cuda.get_device_name(0)})
    from huggingface_hub import snapshot_download
    snapshot = Path(snapshot_download(repo_id=MODEL_ID, revision=MODEL_REVISION))
    if snapshot.is_symlink() or not snapshot.is_dir() or snapshot.resolve().name != MODEL_REVISION: raise RuntimeError('model snapshot identity failed')
    LOCAL_ROOT.mkdir()
    command = [sys.executable, str(WORK/'experiments/run_s1_propagation_sensitivity_multipair.py'), '--output', str(OUTPUT)]
    (LOG/'command.json').write_text(json.dumps({'argv':command,'cwd':str(WORK)},sort_keys=True),encoding='utf-8')
    runner_started = True
    with (LOG/'runner.stdout').open('xb') as out, (LOG/'runner.stderr').open('xb') as err:
        completed = subprocess.run(command,cwd=WORK,stdout=out,stderr=err,check=False)
    stdout=(LOG/'runner.stdout').read_text(encoding='utf-8',errors='replace'); stderr=(LOG/'runner.stderr').read_text(encoding='utf-8',errors='replace')
    print('Runner return code:',completed.returncode)
    if stdout.strip(): print('Runner stdout:\n'+stdout.rstrip())
    if stderr.strip(): print('Runner stderr:\n'+stderr.rstrip())
    if (OUTPUT/'audit.json').is_file(): audit=json.loads((OUTPUT/'audit.json').read_text(encoding='utf-8'))
    expected={'PROPAGATION_CONSTRUCTION_READY':0,'PROPAGATION_CONSTRUCTION_NO_GO':3}
    if audit is None or expected.get(audit.get('status')) != completed.returncode: raise RuntimeError('runner audit missing or inconsistent; inspect stdout above')
    print('Propagation status:',audit['status'])
except BaseException as exc:
    caught=exc; print('Execution failure:',type(exc).__name__,str(exc))
finally:
    BUNDLE.mkdir()
    if LOG.exists(): shutil.copytree(LOG,BUNDLE/'logs')
    if OUTPUT.exists() and OUTPUT.is_dir() and not OUTPUT.is_symlink(): shutil.copytree(OUTPUT,BUNDLE/'output')
    state={'schema':'sstw.s1.propagation.notebook.v1','diagnostic_class':'DIAGNOSTIC_ONLY','run_id':RUN_ID,'source_ref':AUTHORIZED_REF,'runner_started':runner_started,'return_code':None if completed is None else completed.returncode,'audit_status':None if audit is None else audit.get('status'),'failure_type':None if caught is None else type(caught).__name__}
    (BUNDLE/'notebook_state.json').write_text(json.dumps(state,sort_keys=True,separators=(',',':')),encoding='utf-8')
    with zipfile.ZipFile(ARCHIVE,'x',compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(BUNDLE.rglob('*')):
            if path.is_file(): archive.write(path,path.relative_to(BUNDLE))
    with zipfile.ZipFile(ARCHIVE,'r') as archive:
        if archive.testzip() is not None: raise RuntimeError('local ZIP invalid')
    archive_sha=sha256_file(ARCHIVE)
    sidecar={'schema':'sstw.s1.propagation.archive.v1','run_id':RUN_ID,'source_ref':AUTHORIZED_REF,'archive_name':ARCHIVE.name,'archive_size':ARCHIVE.stat().st_size,'archive_sha256':archive_sha,'diagnostic_class':'DIAGNOSTIC_ONLY'}
    SIDECAR.write_text(json.dumps(sidecar,sort_keys=True,separators=(',',':')),encoding='utf-8')
    shutil.copyfile(ARCHIVE,DRIVE_ARCHIVE); shutil.copyfile(SIDECAR,DRIVE_SIDECAR)
    if DRIVE_ARCHIVE.stat().st_size != ARCHIVE.stat().st_size or sha256_file(DRIVE_ARCHIVE) != archive_sha: raise RuntimeError('Drive archive readback mismatch')
    if DRIVE_SIDECAR.read_text(encoding='utf-8') != SIDECAR.read_text(encoding='utf-8'): raise RuntimeError('Drive sidecar readback mismatch')
    with zipfile.ZipFile(DRIVE_ARCHIVE,'r') as archive:
        if archive.testzip() is not None: raise RuntimeError('Drive ZIP invalid')
    print('Packaged:',DRIVE_ARCHIVE,archive_sha)
if caught is not None: raise caught
